In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error


In [22]:
df = pd.read_csv("NFLAnalysis.csv")

#Clean up data
df.columns = df.columns.str.strip()
df = df.replace("—", np.nan)
df = df.replace({',': ''}, regex=True)

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')


C:\Users\9luke\AppData\Local\Temp\ipykernel_55520\1934971282.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [25]:
#These are the variables we use to make the prediction
feature_cols = [
    "Age","Height","Weight","Arm",
    "40-yd","10-yd","Shuttle","3-cone",
    "Vert","Broad","Bench",
    "C Receptions","C Yards","C TDs"]

X = df[feature_cols]

#Predicting NFL production
targets = ["N Yards", "N Receptions", "N TDs"]
models = {}

In [15]:
for t in targets:
    y = df[t]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("gbr", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    models[t] = model

    preds = model.predict(X_test)
    print(f"\nModel for {t}")
    print("R²:", round(r2_score(y_test, preds), 3))
    print("MAE:", round(mean_absolute_error(y_test, preds), 1))



Model for N Yards
R²: -1.029
MAE: 568.0

Model for N Receptions
R²: -0.654
MAE: 41.1

Model for N TDs
R²: 0.178
MAE: 3.4


In [16]:
def get_user_prospect():
    print("\nEnter new prospect data (leave blank if unknown):")
    
    data = {}
    for col in feature_cols:
        val = input(f"{col}: ")
        data[col] = float(val) if val != "" else np.nan
    
    return pd.DataFrame([data])

new_player = get_user_prospect()



Enter new prospect data (leave blank if unknown):


Age:  23
Height:  76
Weight:  210
Arm:  33
40-yd:  4.55
10-yd:  1.58
Shuttle:  4.22
3-cone:  6.95
Vert:  31
Broad:  9009
Bench:  12
C Receptions:  100
C Yards:  1000
C TDs:  10


In [18]:
pred_yards = models["N Yards"].predict(new_player)[0]
pred_recs  = models["N Receptions"].predict(new_player)[0]
pred_tds   = models["N TDs"].predict(new_player)[0]

print("\n--- NFL Projection (First 3 Years) ---")
print("Projected Yards:", int(pred_yards))
print("Projected Receptions:", int(pred_recs))
print("Projected TDs:", int(pred_tds))



--- NFL Projection (First 3 Years) ---
Projected Yards: 178
Projected Receptions: 19
Projected TDs: 5
